# Zone-A LSTM Autoencoder (v3) — time windows, one code per second

Same pipeline, data files and style as v2, with the legacy blocks removed:

* **One sample = one window** of `WINDOW` consecutive seconds × all SubFeatures, shape `(samples, timesteps, features)` = `(N, 10, 64)`. Files are kept in time order (no shuffling); the baseline is split chronologically 70/15/15 and the scaler is fit on the train block only.
* **One code per second.** The encoder LSTM compresses each second's 64 features to `EMBEDDING_DIM = 51` numbers (80% kept); decoder step *t* reads the encoder's hidden state at step *t*: codes = h_1 … h_10, shape `(N, 10, 51)`.
* **Training:** MSE only, Adam (lr 1e-3, no weight decay) — Adagrad with weight_decay 0.01 underfit ~5x.
* **Verification:** cert-rnn with `decoder_input="sequence"`, linked (Tier-1) gates on every pass, τ = the deployed `THRESHOLD`, plus a fixed-ε check mode.

In [ ]:
import torch

import psutil
import threading
import os

import copy
import numpy as np
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from matplotlib import rc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, MaxAbsScaler, RobustScaler

from torch import nn, optim, backends
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F

import sys
for _p in ('/home/verivital/Anne/sandia/cert-rnn-py/src',
           '/ascldap/users/amtumli/dev/cert-rnn-py/src'):
    if os.path.isdir(_p):
        sys.path.insert(0, _p)
        break
import cert_rnn
from cert_rnn import LSTMAutoencoder, ReconErrorSpec

sns.set(style='whitegrid', palette='muted', font_scale=1.2)
COLORS_PALETTE = ["#01BEFE", "#FFDD00", "#FF7D00", "#FF006D", "#ADFF02", "#8F00FF"]
sns.set_palette(sns.color_palette(COLORS_PALETTE))
rcParams['figure.figsize'] = 12, 8

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [ ]:
device = torch.device("cpu")

## Loading Data

In [ ]:
# Load the files into Pandas data frames
# Rows are consecutive seconds, so each file is kept SEPARATE and IN FILE ORDER
# (no concat + shuffle: that would destroy the time order the LSTM needs).
train_fname = "SubFeatures_baseline3hr_zoneA.csv"
test_fname = "SubFeatures_cyberphysical_0_1_zoneA.csv"
labels_fname = "CyPhyScenarioLabels_2Class_0_1_zoneA.csv"

WINDOW = 10                  # seconds per sample = LSTM sequence length (timesteps)
STRIDE = 1                   # seconds between window starts (sliding windows)
SPLIT = (0.70, 0.15, 0.15)   # chronological train / val / test split of the baseline file

base_raw = pd.read_csv(train_fname)   # normal-only baseline
scen_raw = pd.read_csv(test_fname)    # cyber-physical scenario (normal + abnormal)
labels = pd.read_csv(labels_fname)

# Append a label column to the baseline where every label is 0 (normal class)
base_raw['target'] = 0

# Merge the scenario with its labels on 'timestamp' (left merge keeps row order)
n_scen = len(scen_raw)
scen_raw = scen_raw.merge(labels, on='timestamp', how='left')
assert len(scen_raw) == n_scen, "duplicate timestamps in the labels file"
print(f"Number of missing labels in test after merge: {scen_raw['labels'].isna().sum()}")
scen_raw['labels'] = scen_raw['labels'].fillna(1)   # unlabeled = abnormal
scen_raw = scen_raw.rename(columns={'labels': 'target'})

for d in (base_raw, scen_raw):
    if 'Unnamed: 0' in d.columns:
        d.drop('Unnamed: 0', axis=1, inplace=True)

FEATURES = [c for c in base_raw.columns if c not in ('timestamp', 'target')]
assert FEATURES == [c for c in scen_raw.columns if c not in ('timestamp', 'target')]

# df is kept for the EDA cells below (time order preserved, not used for training)
df = pd.concat([base_raw, scen_raw]).drop('timestamp', axis=1).reset_index(drop=True)

print("Number of features:", len(FEATURES))
print("Baseline rows:", len(base_raw), " Scenario rows:", len(scen_raw))
print("Dataframe Head: ")
print(df.head())

CLASS_NORMAL = 0
class_names = ['Normal', 'Abnormal']

In [ ]:
# Check for NaN values
nan_mask = df.isna()


# Check if any NaN values exist in the DataFrame
if nan_mask.any().any():
    print("\nDataFrame contains NaN values.")
else:
    print("\nDataFrame does not contain any NaN values.")

In [ ]:
# Display rows with NaN values
rows_with_nan = df[nan_mask.any(axis=1)]
print("\nRows with NaN values:")
print(len(rows_with_nan))

In [ ]:
df.dropna(inplace=True)

In [ ]:
# Check for NaN values
nan_mask = df.isna()


# Check if any NaN values exist in the DataFrame
if nan_mask.any().any():
    print("\nDataFrame contains NaN values.")
else:
    print("\nDataFrame does not contain any NaN values.")

In [ ]:
# Split the baseline by TIME first (earliest 70% = train), then fit the scaler
# on the TRAIN block only. The same scaler is applied to everything else.
n = len(base_raw)
i_tr, i_va = int(SPLIT[0] * n), int((SPLIT[0] + SPLIT[1]) * n)
base_train_raw = base_raw.iloc[:i_tr]
base_val_raw = base_raw.iloc[i_tr:i_va]
base_test_raw = base_raw.iloc[i_va:]

cols_to_normalize = pd.Index(FEATURES)
scaler = MinMaxScaler()
scaler.fit(base_train_raw[FEATURES])     # NaNs are ignored by the fit

def scale(frame):
    out = frame.copy()
    out[FEATURES] = scaler.transform(out[FEATURES])   # NaNs pass through; windows with NaNs are skipped later
    return out

df[FEATURES] = scaler.transform(df[FEATURES])   # scaled copy for the EDA plots

print(df.head())
print(cols_to_normalize)

In [ ]:
df.min()

In [ ]:
import pickle

# Saving the objects:
with open('zoneA_scaler_timewin.pkl', 'wb') as f:
    pickle.dump([scaler], f)

## EDA

In [ ]:
print("Examples for each class: ")
print(df.target.value_counts())
print(" ")

print("Plotting the above results - Visualization")

df.target.value_counts().plot(kind='bar', figsize=(7, 6), rot=0);
plt.xlabel("Class", labelpad=14)
plt.ylabel("# Normal/Abnormal Packets", labelpad=14)
plt.title("Normal/Abnormal Instances", y=1.02);

In [ ]:
pd.options.display.max_columns = None

df1 = df[df.isna().any(axis=1)]
display(df1)
df1 = df1.dropna()
df1 = df1[df1.isna().any(axis=1)]
display(df1)

In [ ]:
# Averaged (smoothed out with one standard deviation on top and bottom of it) Time Series for each class
def plot_time_series_class(data, class_name, ax, n_steps=2):
    time_series_df = pd.DataFrame(data)

    smooth_path = time_series_df.rolling(n_steps).mean()
    path_deviation = 2 * time_series_df.rolling(n_steps).std()
    
    under_line = (smooth_path - path_deviation)[0]
    over_line = (smooth_path + path_deviation)[0]

    ax.plot(smooth_path, linewidth=4)
    ax.fill_between(
        path_deviation.index,
        under_line,
        over_line,
        alpha=.125
    )
    ax.set_title(class_name + ' Data', fontsize = 32)
    
    ax.xaxis.set_tick_params(labelsize=25)
    ax.yaxis.set_tick_params(labelsize=25)
    ax.set_xlabel('Data Sample', fontsize=32)
    ax.set_ylabel('Avg Time Series', fontsize=32)

classes = df.target.unique()

fig, axs = plt.subplots(
    nrows=len(classes) // 3+1,
    ncols=3,
    sharey=True,
    figsize=(14, 8)
)

for i,cls in enumerate(classes):
    ax = axs.flat[i]
    print(cls)
    
        
    data = df[df.target == cls] \
        .drop(labels='target', axis=1) \
            .mean(axis=1) \
                .to_numpy()
    plot_time_series_class(data, class_names[i], ax)

fig.delaxes(axs.flat[-1])
fig.tight_layout();

In [ ]:
# One sample = WINDOW consecutive seconds x all features -> tensor (WINDOW, n_features).
# Each LSTM step is one second carrying all features.
# Windows never cross a NaN row, a time gap, or a split boundary.

def time_segments(ts):
    """Segment id per row. A new segment starts wherever the timestamp jumps by
    more than 1.5x the typical step. Unparseable timestamps -> one segment."""
    t = pd.to_numeric(ts, errors='coerce')
    if t.isna().any():
        t = pd.to_datetime(ts, errors='coerce')
        if t.isna().any():
            print("WARNING: could not parse timestamps; time gaps are NOT checked")
            return np.zeros(len(ts), dtype=int)
        t = (t - t.iloc[0]).dt.total_seconds()
    dt = t.diff().to_numpy()[1:]
    if np.any(dt <= 0):
        print("WARNING: timestamps are not strictly increasing - check the row order")
    return np.concatenate([[0], np.cumsum(dt > 1.5 * np.median(dt))])

def make_windows(frame, window=WINDOW, stride=STRIDE):
    X = frame[FEATURES].to_numpy(np.float32)
    y = frame['target'].to_numpy()
    ok = ~np.isnan(X).any(axis=1)
    seg = time_segments(frame['timestamp'].reset_index(drop=True))
    xs, ys = [], []
    for s in range(0, len(X) - window + 1, stride):
        e = s + window
        if ok[s:e].all() and seg[s] == seg[e - 1]:
            xs.append(torch.from_numpy(X[s:e].copy()))
            ys.append(int(y[s:e].max()))   # abnormal if ANY second in the window is abnormal
    return xs, np.array(ys, dtype=int)

train_dataset, _ = make_windows(scale(base_train_raw))
val_dataset, _ = make_windows(scale(base_val_raw))
base_test_windows, _ = make_windows(scale(base_test_raw))
scen_windows, scen_y = make_windows(scale(scen_raw))

# Normal test = held-out baseline windows + fully-normal scenario windows
test_normal_dataset = base_test_windows + [w for w, l in zip(scen_windows, scen_y) if l == 0]
test_anomaly_dataset = [w for w, l in zip(scen_windows, scen_y) if l == 1]

seq_len, n_features = train_dataset[0].shape   # (WINDOW, number of features)
print(f"Input shape: (samples, timesteps, features) = ({len(train_dataset)}, {seq_len}, {n_features})")
print(f"train {len(train_dataset)} | val {len(val_dataset)} | "
      f"test normal {len(test_normal_dataset)} | test anomaly {len(test_anomaly_dataset)}")

In [ ]:
print(len(train_dataset))
print(np.shape(train_dataset[0]))
np.asarray(train_dataset[0]).ravel()[:50]

## LSTM-Based Autoencoder

In [ ]:
EMBEDDING_DIM = 51   # per-second code: 64 features -> 51 (80% kept)

class Encoder(nn.Module):
    def __init__(self, seq_len, n_features, embedding_dim=EMBEDDING_DIM):
        super(Encoder, self).__init__()

        self.seq_len, self.n_features = seq_len, n_features
        self.embedding_dim, self.hidden_dim = embedding_dim, embedding_dim

        self.rnn1 = nn.LSTM(
            input_size=n_features,
            hidden_size=self.hidden_dim,
            num_layers=1,
            batch_first=True
        )

    def forward(self, x):
        # Dynamically determine batch size from input tensor
        batch_size = x.shape[0]

        # Reshape input tensor to match expected dimensions
        x = x.reshape((batch_size, self.seq_len, self.n_features))

        # One code per second: the hidden state at EVERY step, (batch, seq_len, embedding_dim)
        codes, (_, _) = self.rnn1(x)

        return codes

In [ ]:
class Decoder(nn.Module):
    def __init__(self, seq_len, input_dim=EMBEDDING_DIM, n_features=1):
        super(Decoder, self).__init__()

        self.seq_len, self.input_dim = seq_len, input_dim
        self.hidden_dim, self.n_features = input_dim, n_features

        self.rnn1 = nn.LSTM(
            input_size=input_dim,
            hidden_size=self.hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.output_layer = nn.Linear(self.hidden_dim, n_features)

    def forward(self, codes):
        # Decoder step t reads code t (the encoder's hidden state at second t)
        batch_size = codes.shape[0]
        x, (_, _) = self.rnn1(codes)
        x = x.reshape((batch_size, self.seq_len, self.hidden_dim))

        return self.output_layer(x)

In [ ]:
class RecurrentAutoencoder(nn.Module):
    def __init__(self, seq_len, n_features, embedding_dim=EMBEDDING_DIM):
        super(RecurrentAutoencoder, self).__init__()

        self.encoder = Encoder(seq_len, n_features, embedding_dim).to(device)
        self.decoder = Decoder(seq_len, embedding_dim, n_features).to(device)
        self.code = None

    def forward(self, x):
        x = self.encoder(x)
        self.code = x          # (batch, seq_len, embedding_dim)
        x = self.decoder(x)
        return x

In [ ]:
def train_model(model, train_dataset, val_dataset, n_epochs, lr, weight_decay, optimizer):

  global BATCH_SIZE

  optimizer = optimizer(model.parameters(), lr=lr, weight_decay=weight_decay)
  criterion = nn.MSELoss().to(device)

  history = dict(train=[], val=[])

  best_model_wts = None
  best_loss = 10000.0
  clip_value = 5

  for epoch in range(1, n_epochs + 1):

    model = model.train()

    train_losses = []

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    for batch_idx, batch in enumerate(train_loader):
      seq_true = batch
      optimizer.zero_grad()

      seq_true = seq_true.to(device)
      if seq_true.shape[0] == BATCH_SIZE:

          seq_pred = model(seq_true)
          loss = criterion(seq_pred, seq_true)

          loss.backward()
          if loss.isnan():
                print("*** None!!! ***")

          torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)
          optimizer.step()

          train_losses.append(loss.item())

    val_losses = []
    model = model.eval()

    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

    with torch.no_grad():
      for batch_idx, batch in enumerate(val_loader):

        seq_true = batch
        if seq_true.shape[0] == BATCH_SIZE:

            seq_true = seq_true.to(device)
            seq_pred = model(seq_true)

            loss = criterion(seq_pred, seq_true)
            val_losses.append(loss.item())

    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)

    history['train'].append(train_loss)
    history['val'].append(val_loss)

    if val_loss < best_loss:
      best_loss = val_loss
      best_model_wts = copy.deepcopy(model.state_dict())

    print(f'Epoch {epoch}: train loss {train_loss} val loss {val_loss}')

  model.load_state_dict(best_model_wts)
  return model.eval(), history

In [ ]:
global BATCH_SIZE

best_model = None
best_history = {'train':[], 'val':[]}
best_history_avg_window = 10000

best_bs = None
best_lr = None
best_wd = None
best_opt = None
best_epochs = None

for epochs in [100]:
    for BATCH_SIZE in [16]:
        for learning_rate in [1e-3]:
            for weight_decay in [0.0]:
                for optimizer in [torch.optim.Adam]:

                    print('--------------------------------------------------------------')
                    print('-----------------------Training Information-------------------')
                    print(' ')
                    print('Optimizer: ', optimizer)
                    print('Weight Decay: ', weight_decay)
                    print('Learning Rate: ', learning_rate)
                    print('Batch Size: ', BATCH_SIZE)
                    print(' ')
                    print(' ')

                    print('Starting LSTM AE Model Training...')
                    print(seq_len)
                    model = RecurrentAutoencoder(seq_len, n_features, EMBEDDING_DIM)
                    model = model.to(device)

                    model, history = train_model(
                      model,
                      train_dataset,
                      val_dataset,
                      n_epochs=epochs,
                      lr=learning_rate,
                      weight_decay=weight_decay,
                      optimizer=optimizer
                    )

                    """
                    Saving the LSTM AE with the best hyperparameters configuration

                    Idea: Check the average validation loss of the last K epochs.
                          We should choose the model with the lowest validation loss,
                          as this indicates the best performance on unseen data.
                    """
                    window_size = 10
                    history_window = sum(history['val'][-window_size:])/window_size

                    if history_window < best_history_avg_window:

                        print('*****************')
                        print('New Best Model...')
                        print('Optimizer: ', optimizer)
                        print('Weight Decay: ', weight_decay)
                        print('Learning Rate: ', learning_rate)
                        print('Batch Size: ', BATCH_SIZE)
                        print('Epochs: ', epochs)
                        print('*****************')

                        best_model = model
                        best_history['train'] = history['train']
                        best_history['val'] = history['val']
                        best_history_avg_window = history_window

                        best_bs = BATCH_SIZE
                        best_lr = learning_rate
                        best_wd = weight_decay
                        best_opt = optimizer
                        best_epochs = epochs

                    print('--------------------------------------------------------------')
                    print('--------------------------------------------------------------')
                    print(' ')
                    print(' ')

## Testing Autoencoders

In [ ]:
ax = plt.figure().gca()

ax.plot(best_history['train'], linewidth=4)
ax.plot(best_history['val'], linewidth=4)

plt.ylabel('Loss', fontsize=32)
plt.xlabel('Epoch', fontsize=32)
ax.xaxis.set_tick_params(labelsize=25)
ax.yaxis.set_tick_params(labelsize=25)

plt.legend(['training loss', 'validation loss'], fontsize=30)
plt.title('Best Model - Loss over training epochs', fontsize=32)
plt.show(); 

In [ ]:
import pickle
with open('zone_A_LSTM_AE_train.pkl', 'wb') as file:
    # Step 4: Use pickle.dump() to save the list
    pickle.dump(best_history['train'], file) 

In [ ]:
with open('zone_A_LSTM_AE_val.pkl', 'wb') as file:
    # Step 4: Use pickle.dump() to save the list
    pickle.dump(best_history['val'], file) 

In [ ]:
MODEL_PATH = 'zoneA_ae_seqcode_h51.pth'

torch.save(best_model, MODEL_PATH)

In [ ]:
def predict(model, dataset):
    predictions, losses = [], []
    predictions_data_points = []
    real_data_points = []
    loss_data_points = []
    code_data_points = []
    predictions_flatten = []
    initial_features_flatten = []

    criterion = nn.MSELoss().to(device)
    with torch.no_grad():
        model = model.eval()

        # Iterate over each data point in the dataset
        for idx in range(len(dataset)):
            # Extract a single data point
            data_point = dataset[idx]

            # Convert to PyTorch tensor if necessary
            if isinstance(data_point, list):
                data_point = torch.tensor(data_point, dtype=torch.float32)
            elif isinstance(data_point, np.ndarray):
                data_point = torch.tensor(data_point, dtype=torch.float32)

            # Reshape the single data point to simulate a batch of size 1
            data_point = data_point.unsqueeze(0).to(device)  # Add batch dimension

            # Forward pass
            seq_pred = model(data_point)

            # Collect predictions and real data
            predictions_data_points.append(seq_pred[0])  # Remove batch dimension
            real_data_points.append(data_point[0])  # Remove batch dimension

            # Collect additional information
            code_data_points.append(model.code[0].cpu().numpy().flatten())  # Sending it to CPU
            predictions_flatten.append(seq_pred[0].cpu().numpy().flatten())  # Sending it to CPU
            initial_features_flatten.append(data_point[0].cpu().numpy().flatten())  # Sending it to CPU

            # Compute loss
            loss = criterion(seq_pred.cpu(), data_point.cpu())  # Sending to CPU
            loss_data_points.append(loss.item())

            # Flatten predictions for overall collection
            predictions.append(seq_pred.cpu().numpy().flatten())  # Sending to CPU
            losses.append(loss.item())

    return predictions, losses, predictions_data_points, real_data_points, loss_data_points, predictions_flatten, code_data_points, initial_features_flatten

In [ ]:
BATCH_SIZE = 1
predictions, losses, _, _, losses_data, _, _, _ = predict(best_model, train_dataset)

print("Number of Predictions:", len(predictions))

ax = sns.distplot(losses_data, bins=90, kde=True)
ax.set(xlabel='RE', ylabel='Normal Data Density')
sns.set(font_scale=3)

In [ ]:
THRESHOLD = np.percentile(losses, 99)
THRESHOLD

In [ ]:
# Normal Data Predictions [~100%]
predictions, pred_losses,  _, _, losses_data, _, _, _= predict(best_model, test_normal_dataset)
correct = sum(l <= THRESHOLD for l in losses_data)
print(f'Correct Normal predictions: {correct}/{len(losses_data)}') 

In [ ]:
# Abnormal Data Predictions 
anomaly_dataset = test_anomaly_dataset[:len(test_normal_dataset)]
predictions, pred_losses,  _, _, losses_data, _, _, _= predict(best_model, anomaly_dataset)
ax = sns.distplot(losses_data, bins=90, kde=True);

ax.set(xlabel='RE', ylabel='Abnormal Data Density')
sns.set(font_scale=3) 

In [ ]:
# (~100%)
correct = sum(l > THRESHOLD for l in losses_data)
print(f'Correct anomaly predictions: {correct}/{len(losses_data)}') 

In [ ]:
# Detection metrics over ALL test windows (normal + abnormal) at THRESHOLD
from sklearn.metrics import roc_auc_score

_, _, _, _, normal_losses, _, _, _ = predict(best_model, test_normal_dataset)
_, _, _, _, anomaly_losses, _, _, _ = predict(best_model, test_anomaly_dataset)
y_true = np.r_[np.zeros(len(normal_losses)), np.ones(len(anomaly_losses))]
y_score = np.r_[normal_losses, anomaly_losses]
y_pred = y_score > THRESHOLD

tp = int((y_pred & (y_true == 1)).sum()); fp = int((y_pred & (y_true == 0)).sum())
fn = int((~y_pred & (y_true == 1)).sum()); tn = int((~y_pred & (y_true == 0)).sum())
precision = tp / max(tp + fp, 1); recall = tp / max(tp + fn, 1)
f1 = 2 * precision * recall / max(precision + recall, 1e-12)
print(f"windows: {len(normal_losses)} normal, {len(anomaly_losses)} abnormal")
print(f"true positive rate (recall) {recall:.3f} | false alarm rate {fp / max(fp + tn, 1):.3f}")
print(f"precision {precision:.3f} | F1 {f1:.3f} | ROC-AUC {roc_auc_score(y_true, y_score):.3f}")

## Threshold analysis

`THRESHOLD` (99th percentile of **training** scores) decides every alarm and is also τ for verification. This section checks whether it sits in the right place:

* **Score distributions** — validation, normal test (baseline block vs scenario-normal) and abnormal windows, with the gap between the highest normal and the lowest abnormal score. A ROC-AUC of 1.0 means such a gap exists: a threshold inside it catches every anomaly with no false alarm.
* **Candidate thresholds from validation only** (never tuned on the test set), each evaluated on the test windows.
* **Where the false alarms are** — baseline vs scenario, and how far they sit from the nearest abnormal second.
* **Events, not windows** — overlapping windows share seconds, so results are also counted per abnormal *event* (contiguous abnormal seconds) and per alarm *episode* (consecutive alarming windows), including a "k consecutive alarming windows" rule.

`THRESHOLD_CHOICE` at the end selects the threshold used from here on (verification τ included); the default keeps the training 99th percentile.

In [ ]:
# Threshold analysis: score distributions, validation-based candidates, false-alarm locations, events
def window_starts(frame, window=WINDOW, stride=STRIDE):
    """Start rows of the windows make_windows keeps (same validity rules), in the same order."""
    X = frame[FEATURES].to_numpy(np.float32)
    ok = ~np.isnan(X).any(axis=1)
    seg = time_segments(frame['timestamp'].reset_index(drop=True))
    return np.array([s for s in range(0, len(X) - window + 1, stride)
                     if ok[s:s + window].all() and seg[s] == seg[s + window - 1]])

def runs(mask):
    """(start, end) index pairs of consecutive True runs, end exclusive."""
    m = np.r_[False, np.asarray(mask, bool), False]
    d = np.flatnonzero(np.diff(m.astype(int)))
    return list(zip(d[::2], d[1::2]))

def time_runs(mask, starts):
    """Runs of alarming windows that are also consecutive in TIME (starts STRIDE apart)."""
    order = np.argsort(starts)
    a, st = np.asarray(mask, bool)[order], np.asarray(starts)[order]
    out = []
    for s0, e0 in runs(a):
        cut = [s0] + [s0 + 1 + j for j in np.flatnonzero(np.diff(st[s0:e0]) != STRIDE)] + [e0]
        out += [(order[x:y], y - x) for x, y in zip(cut[:-1], cut[1:])]
    return out

_, _, _, _, val_losses, _, _, _ = predict(best_model, val_dataset)
_, _, _, _, train_losses, _, _, _ = predict(best_model, train_dataset)
val_losses, train_losses = np.asarray(val_losses), np.asarray(train_losses)
n_base = len(base_test_windows)
base_starts = window_starts(scale(base_test_raw))
assert len(base_starts) == n_base
normal_losses_arr, anomaly_losses_arr = np.asarray(normal_losses), np.asarray(anomaly_losses)
base_normal = normal_losses_arr[:n_base]                     # held-out baseline block
scen_normal = normal_losses_arr[n_base:]                     # scenario file, normal windows

# 1) the gap between normal and abnormal test scores
max_normal, min_abnormal = normal_losses_arr.max(), anomaly_losses_arr.min()
print(f"highest normal test score  {max_normal:.4g}  (baseline {base_normal.max():.4g}, "
      f"scenario-normal {scen_normal.max() if len(scen_normal) else float('nan'):.4g})")
print(f"lowest abnormal test score {min_abnormal:.4g}")
print("gap: " + (f"YES -- any threshold in ({max_normal:.4g}, {min_abnormal:.4g}) separates them perfectly"
                 if min_abnormal > max_normal else
                 f"NO -- {int((anomaly_losses_arr <= max_normal).sum())} abnormal windows score below "
                 f"the highest normal one"))

# 2) scenario labels per second and window starts -> events and distance to events
scen_scaled = scale(scen_raw).reset_index(drop=True)
scen_starts = window_starts(scen_scaled)
assert len(scen_starts) == len(scen_windows)
lab = scen_scaled['target'].to_numpy().astype(int)
events = runs(lab == 1)
abn_sec = np.flatnonzero(lab == 1)
normal_starts = scen_starts[scen_y == 0]
abnormal_starts = scen_starts[scen_y == 1]

def dist_to_event(s):
    """Seconds between window [s, s+WINDOW) and the nearest abnormal second (0 = overlaps)."""
    if len(abn_sec) == 0:
        return np.inf
    e = s + WINDOW - 1
    j = np.searchsorted(abn_sec, s)
    near = abn_sec[max(j - 1, 0):j + 1]
    return int(min(0 if s <= a <= e else (s - a if a < s else a - e) for a in near))

def evaluate(thr, k=1):
    """Test metrics at threshold thr; with k > 1 an alarm needs k consecutive alarming windows."""
    def alarms(scores, starts):
        a = np.asarray(scores) > thr
        if k == 1:
            return a
        out = np.zeros_like(a)
        for idx, n in time_runs(a, starts):
            if n >= k:
                out[idx[k - 1:]] = True      # alarm raised from the k-th consecutive window on
        return out
    fa_base = alarms(base_normal, base_starts)
    fa_scen = alarms(scen_normal, normal_starts)
    det = alarms(anomaly_losses_arr, abnormal_starts)
    tp = int(det.sum()); fp = int(fa_base.sum() + fa_scen.sum())
    ev_hit = sum(bool(np.any(det[(abnormal_starts + WINDOW > s) & (abnormal_starts < e)])) for s, e in events)
    episodes = len(time_runs(fa_base, base_starts)) + len(time_runs(fa_scen, normal_starts))
    prec = tp / max(tp + fp, 1); rec = tp / max(len(det), 1)
    return {"threshold": thr, "recall": rec,
            "false alarm rate": fp / max(len(normal_losses_arr), 1),
            "FA rate baseline": fa_base.mean() if len(fa_base) else np.nan,
            "FA rate scenario-normal": fa_scen.mean() if len(fa_scen) else np.nan,
            "false-alarm episodes": episodes, "precision": prec,
            "F1": 2 * prec * rec / max(prec + rec, 1e-12),
            "events detected": f"{ev_hit}/{len(events)}"}

# 3) candidate thresholds -- from TRAIN/VALIDATION scores only
candidates = {
    "train p99 (current)": float(np.percentile(train_losses, 99)),
    "val p99": float(np.percentile(val_losses, 99)),
    "val p99.9": float(np.percentile(val_losses, 99.9)),
    "val max": float(val_losses.max()),
    "val max x 1.1": float(1.1 * val_losses.max()),
}
print(f"\n{len(events)} abnormal events in the scenario file "
      f"({len(abnormal_starts)} abnormal windows overlap them)")
display(pd.DataFrame({name: evaluate(t) for name, t in candidates.items()}).T)

# 4) where the false alarms are at the current THRESHOLD
fa_scen_mask = scen_normal > THRESHOLD
d = np.array([dist_to_event(s) for s in normal_starts[fa_scen_mask]])
print(f"\nfalse alarms at THRESHOLD={THRESHOLD:.4g}: {int((base_normal > THRESHOLD).sum())} in the baseline "
      f"test block, {int(fa_scen_mask.sum())} in scenario-normal windows")
if len(d):
    for lo, hi in ((0, 5), (5, 30), (30, 120), (120, np.inf)):
        print(f"  scenario false alarms {lo:>3}-{hi if np.isfinite(hi) else 'inf':>3} s from the nearest "
              f"abnormal second: {int(((d >= lo) & (d < hi)).sum())}")

# 5) alarm rule: k consecutive alarming windows
print("\nk consecutive alarming windows before an alarm (current THRESHOLD):")
display(pd.DataFrame({f"k={k}": evaluate(THRESHOLD, k) for k in (1, 3, 5, 10)}).T)

# 6) distributions
plt.figure(figsize=(12, 5))
allv = np.r_[val_losses, normal_losses_arr, anomaly_losses_arr]
bins = np.logspace(np.log10(allv[allv > 0].min()), np.log10(allv.max()), 80)
plt.hist(val_losses, bins=bins, alpha=0.5, density=True, label="validation (normal)")
plt.hist(base_normal, bins=bins, alpha=0.5, density=True, label="test normal - baseline")
if len(scen_normal):
    plt.hist(scen_normal, bins=bins, alpha=0.5, density=True, label="test normal - scenario")
plt.hist(anomaly_losses_arr, bins=bins, alpha=0.5, density=True, label="test abnormal")
for (name, t), ls in zip(candidates.items(), ["-", "--", ":", "-.", (0, (1, 3))]):
    plt.axvline(t, color="k", ls=ls, lw=1.2, label=name)
plt.xscale("log"); plt.xlabel("reconstruction score (window MSE)"); plt.ylabel("density")
plt.title("Score distributions and candidate thresholds"); plt.legend(fontsize=9)
plt.show()

# 7) threshold used from here on (verification tau included)
THRESHOLD_CHOICE = "train p99 (current)"
THRESHOLD = candidates[THRESHOLD_CHOICE]
print(f"THRESHOLD = {THRESHOLD:.6g}  ({THRESHOLD_CHOICE})")

## Verification Pipeline v3 — Cert-RNN+ on the per-step-code model

Same spec as v2 (reconstruction score ≤ τ over the L∞ ball), updated for this model:

1. **Per-step codes.** The model is converted with `decoder_input="sequence"`: decoder step *t* reads the encoder's hidden state at step *t*. Encoder + decoder is then a stacked LSTM with a per-step head, built from the same sound transformers as before.
2. **Linked (Tier-1) gates on every pass** (the driver default) and the joint quadratic score bound.
3. **τ = `THRESHOLD`**, the deployed alarm, so the certificates are about the detector that actually runs. The anchor is the **median-score** validation window (a typical window, not the easiest one).
4. **Two ways to run:** the certified-radius search (largest ε that is proven) or a **fixed-ε check** (`--eps`): one pass per attack at exactly that ε, labelled VERIFIED / COUNTEREXAMPLE / UNKNOWN.

The heavy computation runs outside the notebook:

| Step | Where | What |
|---|---|---|
| 1 | notebook | convert model → preflight/smoke → save a *job file* |
| 2 | **tmux** | `verify_zoneA_v2.py` runs the stages, checkpointing each to JSON (SSH-drop-safe, resumable) |
| 3 | notebook | load the results JSON → summary tables + figures |

In [ ]:
# If the training cells above were skipped, reload the trained model from disk.
MODEL_PATH = 'zoneA_ae_seqcode_h51.pth'
if 'best_model' not in globals() or best_model is None:
    best_model = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
model = best_model
model.eval()
print(model.__class__.__name__, "ready")

In [ ]:
# Convert the trained torch model -> cert-rnn (per-step codes: decoder_input="sequence")
def cell(r):
    c = nn.LSTMCell(r.input_size, r.hidden_size).to(r.weight_ih_l0.dtype)
    with torch.no_grad():
        c.weight_ih.copy_(r.weight_ih_l0); c.weight_hh.copy_(r.weight_hh_l0)
        c.bias_ih.copy_(r.bias_ih_l0);     c.bias_hh.copy_(r.bias_hh_l0)
    return c

ae = LSTMAutoencoder.from_torch(
    [cell(model.encoder.rnn1)],   # encoder: 1 layer
    [cell(model.decoder.rnn1)],   # decoder: 1 layer
    model.decoder.output_layer,
    decoder_input="sequence")

print("captured:", ae.encoder["L"], "enc layers,", ae.decoder["L"], "dec layers",
      " D=", ae.D, " H=", ae.H, " decoder input:", ae.decoder["input"])

In [ ]:
# Threshold tau = the deployed alarm THRESHOLD; anchor = the median-score validation window
X_val = np.stack([np.asarray(val_dataset[i], dtype=np.float64).reshape(-1, ae.D)
                  for i in range(len(val_dataset))])
scores = ae.score(X_val)
tau = float(THRESHOLD)
i = int(np.argsort(scores)[len(scores) // 2])
anchor = X_val[i]
print(f"tau={tau:.5f} anchor#{i} score={scores[i]:.5f} (< tau: {scores[i] < tau}, "
      f"margin {tau - scores[i]:.5f})")

In [ ]:
# Preflight (format/parity) + smoke test (cost forecast) before any long run
report = ae.preflight(anchor, tau=tau, torch_model=model)
print(report)
assert report.ok

smoke = ae.smoke_test(anchor, tau, n_frames=max(2, anchor.shape[0] // 2))
assert smoke["ok"]
print(f"one full-length reach ~{smoke['est_sec_per_reach_full']:.0f}s (box gates)")
print(f"certify multi_frame  ~{smoke['est_certify_multi_frame_s']/60:.1f} min (serial box mode)")
print(f"certify single_frame ~{smoke['est_certify_single_frame_s']/60:.1f} min (serial box mode; "
      f"linked gates cost more per pass, the parallel walk divides by ~n_workers)")

In [ ]:
# Step 1 -- save the verification job: cert-rnn model dicts + anchor + tau.
# The payload is numpy-only, so the script needs neither torch nor the
# model classes defined in this notebook. decoder["input"] = "sequence"
# travels inside the decoder dict.
import pickle

JOB, RESULTS = "verify_job_seqcode_h51.pkl", "verify_results_seqcode_h51.json"
with open(JOB, "wb") as f:
    pickle.dump({"encoder": ae.encoder, "decoder": ae.decoder,
                 "head": ae.head, "anchor": anchor, "tau": tau}, f)
torch.save(model.state_dict(), "lstm_ae_checkpoint_seqcode_h51.pt")  # weights, for the record
print("wrote", JOB, "and lstm_ae_checkpoint_seqcode_h51.pt")

### Batch windows — Option A and Option B

A single anchor says something about **one** window. For headline results the same checks run over many windows:

* **Option A — 10 back-to-back windows:** one continuous **100-second stretch** of the held-out baseline test block (seconds 0–9, 10–19, …, 90–99 from `OPTION_A_START`), so every second of the stretch is checked exactly once.
* **Option B — 100 windows spread across the test set:** a random sample (fixed seed) of all normal test windows.

Windows already above τ at ε = 0 are false alarms as they stand; the driver reports them and does not check them.

In [ ]:
# Batch job files: Option A (10 back-to-back windows) and Option B (100 spread-out windows)
OPTION_A_START = 0      # first second of the 100-second stretch in the baseline test block
N_OPTION_B = 100

base_test_scaled = scale(base_test_raw).reset_index(drop=True)
Xb = base_test_scaled[FEATURES].to_numpy(np.float64)
seg = time_segments(base_test_scaled['timestamp'])
starts_A = [OPTION_A_START + WINDOW * i for i in range(10)]
end_A = starts_A[-1] + WINDOW
assert end_A <= len(Xb), "OPTION_A_START too late for a 100-second stretch"
assert not np.isnan(Xb[starts_A[0]:end_A]).any() and seg[starts_A[0]] == seg[end_A - 1], \
    "NaN or time gap inside this stretch -- choose another OPTION_A_START"
anchors_A = np.stack([Xb[s:s + WINDOW] for s in starts_A])
info_A = [{"name": f"A{i}: test seconds {s}-{s + WINDOW - 1}", "source": "baseline test block",
           "start": s, "timestamp": str(base_test_scaled['timestamp'].iloc[s])}
          for i, s in enumerate(starts_A)]

rng_B = np.random.default_rng(RANDOM_SEED)
idx_B = np.sort(rng_B.choice(len(test_normal_dataset), size=N_OPTION_B, replace=False))
anchors_B = np.stack([np.asarray(test_normal_dataset[i], dtype=np.float64) for i in idx_B])
info_B = [{"name": f"B{j}: test normal window {i}",
           "source": "baseline test block" if i < len(base_test_windows) else "scenario (normal)",
           "index": int(i)} for j, i in enumerate(idx_B)]

JOB_A, RESULTS_A = "verify_job_seqcode_h51_optionA.pkl", "verify_results_seqcode_h51_optionA.json"
JOB_B, RESULTS_B = "verify_job_seqcode_h51_optionB.pkl", "verify_results_seqcode_h51_optionB.json"
for job_file, anc, info in ((JOB_A, anchors_A, info_A), (JOB_B, anchors_B, info_B)):
    with open(job_file, "wb") as f:
        pickle.dump({"encoder": ae.encoder, "decoder": ae.decoder, "head": ae.head,
                     "tau": tau, "anchors": anc, "anchor_info": info}, f)
    s = ae.score(anc)
    print(f"wrote {job_file}: {len(anc)} windows, {int((s > tau).sum())} already above tau "
          f"(score range {s.min():.3g} .. {s.max():.3g}, tau {tau:.3g})")

### Step 2 — run the driver in tmux

The script lives in the cert-rnn repo (`examples/zoneA_lstm_ae/verify_zoneA_v2.py`).

**Attacks** (`--attack`, comma list or `all`):

* `whole` — every reading of every second within ±ε (worst case; covers gradual attacks)
* `seconds` — each single second on its own (spike attack)
* `concentrated` — the seconds in `--seconds` jointly, e.g. `--seconds 4,5,6`

**Two ways to run:**

* **Fixed-ε check** (`--eps 0.001,0.003,...`): one pass per window × attack × gate mode at exactly that ε, all in parallel. Each result is labelled VERIFIED / COUNTEREXAMPLE / UNKNOWN using a concrete attack at the same ε. Counterexamples are saved as `.npy` files next to the results (`--minimize-cex` also saves the fewest-readings version). For a batch job the log ends with a **HEADLINE** count per attack.
* **Certified radius**: the largest ε that is proven, per attack (stages `box_/zono_single_frame`, `box_/zono_/zono_joint_multi`, `box_/zono_/zono_joint_seconds_4-5-6`). Linked gates run on every pass.

ε is in normalized units (a fraction of each feature's normal range). Each stage checkpoints to the JSON when it finishes; killed runs resume. Detach with `Ctrl-b d`; reattach with `tmux attach -t certify`; watch with `tail -f verify_v3.log`.

In [ ]:
# Print the exact tmux commands for THIS machine (paths resolved)
import cert_rnn, pathlib
script = pathlib.Path(cert_rnn.__file__).resolve().parents[2] \
    / "examples" / "zoneA_lstm_ae" / "verify_zoneA_v2.py"
assert script.exists(), script
workers = max(1, (os.cpu_count() or 2) - 2)
EPS_CHECK = "0.001,0.003,0.01"
CONC_SECONDS = "4,5,6"                     # concentrated attack: these seconds jointly
ATTACKS = "whole,seconds,concentrated"
common = f"--workers {workers} --attack {ATTACKS} --seconds {CONC_SECONDS}"
print("tmux new -s certify")
print(f"cd {os.getcwd()}")
for label, job_file, res_file in (("single anchor", JOB, RESULTS),
                                  ("Option A (10 back-to-back windows)", JOB_A, RESULTS_A),
                                  ("Option B (100 windows)", JOB_B, RESULTS_B)):
    print(f"# {label} -- fixed-eps check:")
    print(f"python {script} --job {job_file} --out {res_file} {common} "
          f"--eps {EPS_CHECK} --minimize-cex 2>&1 | tee -a verify_v3.log")
    print(f"# {label} -- certified radius:")
    print(f"python {script} --job {job_file} --out {res_file} {common} "
          f"2>&1 | tee -a verify_v3.log")
print("# Ctrl-b d to detach; tmux attach -t certify to return")

In [ ]:
# Step 3 -- load the results (rerun any time; stages appear as they finish)
import json

res = json.load(open(RESULTS))
S = res["stages"]
print(f"meta: {res['meta']}\n")
radius_names = [n for n in S if not n.startswith("check_eps_")]
for name in radius_names:
    st = S[name]
    r = st.get("radius")
    print(f"{name:24s}: " + (f"eps_c = {r:.6f}  " if r is not None else "")
          + f"({st['seconds']:.0f}s, {st['done_at']})")

if "zono_single_frame" in S and "box_single_frame" in S:
    pf_z = np.asarray(S["zono_single_frame"]["per_frame"])
    pf_b = np.asarray(S["box_single_frame"]["per_frame"])
    r_z, r_b = S["zono_single_frame"]["radius"], S["box_single_frame"]["radius"]
    print(f"\nsingle-frame eps_c:  Cert-RNN {r_b:.6f}  ->  "
          f"Cert-RNN+ {r_z:.6f}   gain {(r_z / r_b - 1) * 100:+.1f}%")
    assert np.all(pf_z >= pf_b - 1e-12)   # Tier 1 never below the baseline

In [ ]:
# Fixed-eps check results (--eps): verdict per attack and gate mode
def check_rows(stages, window=None):
    rows = []
    for name, st in sorted(stages.items()):
        if not name.startswith("check_eps_"):
            continue
        entries = ([("whole window", st["whole_window"])] if "whole_window" in st else []) + \
                  [(f"second {e['t']}", e) for e in st.get("one_second", [])] + \
                  ([("seconds " + ",".join(map(str, st["concentrated"]["seconds"])), st["concentrated"])]
                   if "concentrated" in st else [])
        for label, e in entries:
            rows.append({**({"window": window} if window is not None else {}),
                         "eps": st["eps"], "attack": label,
                         "concrete attack score": e["attack_score"],
                         "bound (independent gates)": e["box"]["ub_joint"],
                         "verdict (independent)": e["box"]["verdict_joint"],
                         "bound (linked gates)": e["zono"]["ub_joint"],
                         "verdict (linked)": e["zono"]["verdict_joint"],
                         "counterexample": e.get("counterexample", {}).get("file", "")})
    return rows

rows = check_rows(S)
if rows:
    print(f"tau = {tau:.6g}  (bounds use the joint score bound)")
    display(pd.DataFrame(rows))
else:
    print("no fixed-eps check run yet (use --eps)")

In [ ]:
# HEADLINE -- batch results (Option A / Option B): verdict counts per attack and eps
for label, res_file in (("Option A (10 back-to-back windows)", RESULTS_A),
                        ("Option B (100 windows)", RESULTS_B)):
    if not os.path.exists(res_file):
        print(f"{label}: not run yet ({res_file})")
        continue
    rb = json.load(open(res_file))
    rows = []
    for name, sm in sorted(rb.get("summary", {}).items()):
        for fam in ("whole", "seconds", "concentrated"):
            if fam in sm:
                c = sm[fam]
                rows.append({"eps": float(name.replace("check_eps_", "")), "attack": fam,
                             "windows checked": sm["windows"] - sm["already_alarm"],
                             "VERIFIED": c["VERIFIED"], "COUNTEREXAMPLE": c["COUNTEREXAMPLE"],
                             "UNKNOWN": c["UNKNOWN"], "UNSOUND": c["UNSOUND"],
                             "already above tau": sm["already_alarm"]})
    print(f"\n{label}")
    if rows:
        display(pd.DataFrame(rows))
    for name, v in rb.get("summary_radius", {}).items():
        print(f"  {name:28s} certified radius over {v['windows']} windows: "
              f"min {v['min']:.6f} | median {v['median']:.6f} | max {v['max']:.6f}")

In [ ]:
# Counterexamples: re-score with the torch model, show the change in RAW units
import glob

def counterexamples(res_obj):
    """(window, eps, attack label, counterexample dict) for every saved counterexample."""
    out = []
    per_window = ([(None, res_obj["stages"])] if "anchors" not in res_obj else
                  [(int(k), a["stages"]) for k, a in res_obj["anchors"].items()])
    for k, stages in per_window:
        for name, st in stages.items():
            if not name.startswith("check_eps_"):
                continue
            entries = ([("whole window", st["whole_window"])] if "whole_window" in st else []) + \
                      [(f"second {e['t']}", e) for e in st.get("one_second", [])] + \
                      ([("seconds " + ",".join(map(str, st["concentrated"]["seconds"])), st["concentrated"])]
                       if "concentrated" in st else [])
            out += [(k, st["eps"], lab, e["counterexample"]) for lab, e in entries if "counterexample" in e]
    return out

all_cex = []
for label, res_file in (("single anchor", RESULTS), ("Option A", RESULTS_A), ("Option B", RESULTS_B)):
    if os.path.exists(res_file):
        all_cex += [(label,) + c for c in counterexamples(json.load(open(res_file)))]
print(f"{len(all_cex)} counterexamples saved")

if all_cex:
    rows = []
    for label, k, eps, lab, c in all_cex:
        xc = np.load(c["file"])
        with torch.no_grad():
            s_torch = float(((model(torch.as_tensor(xc[None], dtype=torch.float32))
                              - torch.as_tensor(xc[None], dtype=torch.float32)) ** 2).mean())
        rows.append({"run": label, "window": k, "eps": eps, "attack": lab,
                     "score (cert-rnn)": c["score"], "score (torch float32)": s_torch,
                     "crosses tau in torch": s_torch > tau, "readings changed": c["n_changed"],
                     "readings changed (minimized)": c.get("minimized", {}).get("n_changed", "")})
    display(pd.DataFrame(rows))

    # Detail for the first counterexample: raw-unit changes + heat map
    label, k, eps, lab, c = all_cex[0]
    xc = np.load(c.get("minimized", c)["file"])
    anc = anchor if k is None else (anchors_A if label == "Option A" else anchors_B)[k]
    raw_anc, raw_cex = scaler.inverse_transform(anc), scaler.inverse_transform(xc)
    d = raw_cex - raw_anc
    t_idx, f_idx = np.nonzero(np.abs(xc - anc) > 0)
    print(f"\n{label}, window {k}, eps={eps}, attack {lab}: "
          f"{len(t_idx)} readings changed" + (" (minimized)" if "minimized" in c else ""))
    display(pd.DataFrame({"second": t_idx, "SubFeature": [FEATURES[f] for f in f_idx],
                          "anchor (raw)": raw_anc[t_idx, f_idx], "attack (raw)": raw_cex[t_idx, f_idx],
                          "change (raw)": d[t_idx, f_idx],
                          "change (% of normal range)": 100 * (xc - anc)[t_idx, f_idx]})
            .sort_values("change (% of normal range)", key=np.abs, ascending=False).head(20))
    plt.figure(figsize=(12, 3.5))
    plt.imshow((xc - anc).T, aspect="auto", cmap="coolwarm", vmin=-eps, vmax=eps)
    plt.colorbar(label="change (normalized units)")
    plt.xlabel("second in window"); plt.ylabel("feature index")
    plt.title(f"Counterexample: change from the anchor ({lab}, eps={eps})")
    plt.show()

In [ ]:
# Figure: per-second certified radii -- Cert-RNN vs Cert-RNN+ (single_frame)
pf_z = np.asarray(S["zono_single_frame"]["per_frame"])
pf_b = np.asarray(S["box_single_frame"]["per_frame"])
T = len(pf_z)
x = np.arange(T)

plt.figure(figsize=(11, 4.5))
plt.bar(x - 0.2, pf_b, width=0.4, label="Cert-RNN (independent gates)", color="C0")
plt.bar(x + 0.2, pf_z, width=0.4, label="Cert-RNN+ (linked gates)", color="C1")
plt.xlabel("perturbed second t in the window (all features at once)")
plt.ylabel(r"certified radius $\epsilon_c(t)$")
plt.title("Per-second certified radius: baseline vs refined")
plt.legend()
plt.show()

worst = int(np.argmin(pf_z))
print(f"most vulnerable second: t={worst}  "
      f"(eps_c {pf_b[worst]:.6f} -> {pf_z[worst]:.6f})")

In [ ]:
# De-normalized certified tolerances: what eps_c means in RAW units.
# Frame t is SECOND t of the window; single_frame corrupts ALL features at
# that second together, each by up to eps_c(t) of its own normal range.
# MinMax is affine per feature, so eps maps to raw units as eps * (max - min).
import pickle
import pandas as pd

with open("zoneA_scaler_timewin.pkl", "rb") as f:
    (scaler,) = pickle.load(f)

feature_names = list(cols_to_normalize)
raw_range = scaler.data_max_ - scaler.data_min_       # per-feature span
raw_anchor = scaler.inverse_transform(anchor)          # (WINDOW, n_features), raw units

pf = np.asarray(S["zono_single_frame"]["per_frame"])
assert len(pf) == anchor.shape[0] and anchor.shape[1] == len(feature_names)

print("Certified radius per second (all features at that second perturbed together):")
display(pd.DataFrame({"second t": np.arange(len(pf)), "eps_c (norm.)": pf}))

def tolerance_table(eps, anchor_raw_row):
    return pd.DataFrame({
        "SubFeature": feature_names,
        "anchor (raw)": anchor_raw_row,
        "eps (raw units)": eps * raw_range,
        "certified low": anchor_raw_row - eps * raw_range,
        "certified high": anchor_raw_row + eps * raw_range,
    })

t_worst = int(np.argmin(pf))
print(f"Physical tolerance per feature at the most vulnerable second t={t_worst} "
      f"(eps_c={pf[t_worst]:.6f}):")
display(tolerance_table(pf[t_worst], raw_anchor[t_worst]))

for name in ("zono_joint_multi", "zono_multi", "box_multi"):
    if name in S and S[name].get("radius") is not None:
        r = S[name]["radius"]
        print(f"{name}: every feature at every second may move by eps={r:.6f} of its range:")
        display(tolerance_table(r, raw_anchor.mean(axis=0)).rename(
            columns={"anchor (raw)": "anchor (raw, window mean)"}))
        break

In [ ]:
# DIAGNOSTIC: saturation, dilution, and a concrete soundness probe.
# A per-second radius at the bisection CEILING means the search never met a
# failing eps: read it as "radius >= ceiling (saturated)", not a measurement.
ni, ei = res["meta"]["n_iters"], res["meta"]["eps_init"]
ceiling = ei * (2 - 0.5 ** ni)
pf = np.asarray(S["zono_single_frame"]["per_frame"])
print(f"search ceiling = {ceiling:.6f}; seconds saturated: "
      f"{int((pf >= ceiling - 1e-9).sum())}/{len(pf)}")

# Dilution: the score is a mean over N = WINDOW * n_features values and
# single_frame moves n_features of them (one whole second): one corrupted
# second is diluted WINDOW-fold. Ignoring the AE's response, crossing tau
# needs a per-value shift of about |delta| ~ sqrt(WINDOW * (tau - s0)).
T_, D_ = anchor.shape
s0 = float(ae.score(anchor[None])[0])
need = np.sqrt(T_ * (tau - s0))
print(f"s0={s0:.5f}  tau={tau:.5f}  margin={tau - s0:.5f}  N={T_ * D_} ({T_} s x {D_} features)")
print(f"one-second crossing needs |delta| ~ {need:.3f} on every feature "
      f"(normalized units; each feature's normal range is 1.0)")

# Concrete soundness probe: at eps = the smallest certified radius, push every
# second's box to its CORNERS (all +, all -, and random sign patterns) through
# the real model. Every score must stay <= tau, else the certificate is UNSOUND.
rng = np.random.default_rng(0)
eps_chk = float(pf.min())
worst = 0.0
for t in range(T_):
    signs = np.vstack([np.ones(D_), -np.ones(D_), rng.choice([-1.0, 1.0], size=(62, D_))])
    X = np.repeat(anchor[None], len(signs), axis=0)
    X[:, t, :] += eps_chk * signs
    worst = max(worst, float(np.max(ae.score(X))))
print(f"concrete worst one-second corner score at eps={eps_chk:.4f}: "
      f"{worst:.5f} vs tau={tau:.5f}  -> "
      + ("consistent (certificate holds concretely)" if worst <= tau
         else "*** SOUNDNESS VIOLATION -- report it ***"))

### Empirical validation of the certified radii

A certified radius ε_c is a proof: **no** input within ±ε_c of the window (on the readings the attack may change) can raise an alarm. This check tries hard to break that claim with the **deployed torch model**:

* **Perturbations inside ε_c**, only on the readings the attack may change (whole window: all seconds; one-second: that second; concentrated: the chosen seconds): random sign corners of the ε-box (where worst cases usually sit), random interior points, and a gradient attack (PGD) that actively searches for an alarm.
* **Expected result: 0 alarms** at ε_c for every window and attack. Any alarm is a soundness finding and is reported loudly.
* **Contrast at 2 × ε_c:** the same search just outside the proof, to show that alarms do become possible there (the radius is not trivially safe).

Runs on the single anchor and on every Option A / Option B window that has certified-radius results.

In [ ]:
# Empirical validation: perturb within the certified radius -> the model must NOT alarm
N_CORNERS, N_INTERIOR, PGD_RESTARTS, PGD_STEPS = 2000, 1000, 8, 40
rng_v = np.random.default_rng(RANDOM_SEED)

def torch_scores(X):
    """Window scores from the deployed torch model (float32), batched."""
    out = []
    with torch.no_grad():
        for i in range(0, len(X), 1024):
            xb = torch.as_tensor(X[i:i + 1024], dtype=torch.float32)
            out.append(((model(xb) - xb) ** 2).mean(dim=(1, 2)).numpy())
    return np.concatenate(out)

def pgd_max(anc, eps, mask):
    """Largest torch score PGD finds inside the eps-box on the masked readings."""
    a = torch.as_tensor(anc[None], dtype=torch.float32).repeat(PGD_RESTARTS, 1, 1)
    m = torch.as_tensor(mask[None], dtype=torch.float32)
    x = a + eps * m * (2 * torch.rand(a.shape, generator=torch.Generator().manual_seed(RANDOM_SEED)) - 1)
    for _ in range(PGD_STEPS):
        x.requires_grad_(True)
        s = ((model(x) - x) ** 2).mean(dim=(1, 2))
        g, = torch.autograd.grad(s.sum(), x)
        with torch.no_grad():
            x = a + torch.clamp(x + 0.1 * eps * m * g.sign() - a, -eps, eps) * m
    with torch.no_grad():
        return float(((model(x) - x) ** 2).mean(dim=(1, 2)).max())

def probe(anc, eps, mask):
    """(samples tried, max score) over corners + interior + PGD inside the eps-box."""
    k = int(mask.sum())
    corners = rng_v.choice([-1.0, 1.0], size=(N_CORNERS, k))
    corners[0], corners[1] = 1.0, -1.0                  # all +, all -
    inner = rng_v.uniform(-1.0, 1.0, size=(N_INTERIOR, k))
    X = np.repeat(anc[None], N_CORNERS + N_INTERIOR, axis=0)
    X[:, mask.astype(bool)] += eps * np.vstack([corners, inner])
    return len(X) + PGD_RESTARTS, max(float(torch_scores(X).max()), pgd_max(anc, eps, mask))

def attack_radii(stages, T):
    """(attack label, mask, certified eps) = the largest certified radius per attack."""
    out = []
    whole = [stages[n]["radius"] for n in ("box_multi", "zono_multi", "zono_joint_multi") if n in stages]
    if whole:
        out.append(("whole window", np.ones((T, 1)), max(whole)))
    if "zono_single_frame" in stages or "box_single_frame" in stages:
        pf = np.max([stages[n]["per_frame"] for n in ("box_single_frame", "zono_single_frame")
                     if n in stages], axis=0)
        for t in range(T):
            m = np.zeros((T, 1)); m[t] = 1
            out.append((f"second {t}", m, float(pf[t])))
    for n in stages:
        if n.startswith("zono_joint_seconds_"):
            secs = [int(s) for s in n.replace("zono_joint_seconds_", "").split("-")]
            r = max(stages[x]["radius"] for x in (n, n.replace("zono_joint", "zono"),
                                                   n.replace("zono_joint", "box")) if x in stages)
            m = np.zeros((T, 1)); m[secs] = 1
            out.append((f"seconds {','.join(map(str, secs))}", m, r))
    return out

runs = [("single anchor", None, anchor, S)]
for label, res_file, anc_arr in (("Option A", RESULTS_A, anchors_A), ("Option B", RESULTS_B, anchors_B)):
    if os.path.exists(res_file):
        rb = json.load(open(res_file))
        runs += [(label, int(k), anc_arr[int(k)], a["stages"]) for k, a in rb.get("anchors", {}).items()]

rows = []
for label, k, anc, stages in runs:
    T_, D_ = anc.shape
    for att, m1, eps_c in attack_radii(stages, T_):
        if eps_c <= 0:
            continue
        mask = np.repeat(m1, D_, axis=1)
        n_in, s_in = probe(anc, eps_c, mask)
        _, s_out = probe(anc, 2 * eps_c, mask)
        rows.append({"run": label, "window": k, "attack": att, "certified eps": eps_c,
                     "samples within eps": n_in, "max score within eps": s_in,
                     "alarms within eps": int(s_in > tau),
                     "margin to tau": tau - s_in,
                     "max score at 2x eps": s_out, "alarm possible at 2x eps": s_out > tau})

val = pd.DataFrame(rows)
if val.empty:
    print("no certified-radius results yet -- run the radius stages first")
else:
    print(f"tau = {tau:.6g}; {len(val)} (window, attack) pairs; "
          f"{int(val['samples within eps'].sum()):,} perturbed windows scored inside the certified radii")
    display(val.groupby(["run", "attack"], sort=False).agg(
        windows=("certified eps", "size"), min_eps=("certified eps", "min"),
        worst_score_within=("max score within eps", "max"),
        alarms_within=("alarms within eps", "sum"),
        alarm_possible_at_2x=("alarm possible at 2x eps", "sum")).reset_index())
    bad = val[val["alarms within eps"] > 0]
    if len(bad):
        print("*** SOUNDNESS FINDING: a perturbation INSIDE a certified radius raised an alarm ***")
        display(bad)
    else:
        print("PASS: no perturbation inside any certified radius raised an alarm "
              f"(closest approach: margin {val['margin to tau'].min():.3g} below tau)")